# ImpactUFSCar Found — Laboratório de backtesting

**Estratégia:** 75% Momentum 12–1 e 25% baixa volatilidade. **PL inicial fictício:** R$ 100 milhões.

Este notebook é um **roteiro didático** para entender, executar e auditar um backtest. Ele não contém histórico de mercado nem simula retornos fictícios. Primeiro execute a coleta pelo arquivo `rodar_windows.bat` ou pelo comando na célula abaixo. Os retornos só aparecem quando as séries históricas reais tiverem sido baixadas.

> Alerta: resultados obtidos com o universo fixo inicial apresentam viés de sobrevivência. Não use como evidência de um fundo administrado ou como promessa de superar o CDI.


## 1. Hipótese de investimento

A hipótese de Momentum propõe que ações com desempenho relativo superior em meses anteriores podem apresentar persistência de retornos. Usamos preços **ajustados por dividendos e desdobramentos** para evitar distorções.

**Momentum 12–1:**

\[ MOM_i(t)= \frac{P_{i,t-21}}{P_{i,t-252}}-1 \]

**Volatilidade anualizada:**

\[ \sigma_i(t)=\operatorname{std}(r_{i,t-62},...,r_{i,t})\sqrt{252} \]

**Score:**

\[ S_i(t)=0,75\,Pctl(MOM_i(t)) + 0,25\,Pctl(-\sigma_i(t)) \]

O percentil de volatilidade é decrescente; menor volatilidade recebe maior score. Os pesos **75/25** são hipóteses fixadas antes do experimento, não pesos escolhidos para ajustar retroativamente um resultado.


## 2. Execução histórica do modelo

A célula abaixo executa a coleta Yahoo Finance + BCB, gera sinais com defasagem de um pregão e cria a pasta `output`. É preciso estar com a internet disponível. Se já tiver executado o arquivo `.bat`, pode pular para a próxima seção.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

raiz = Path.cwd().resolve()
if raiz.name == 'notebooks':
    raiz = raiz.parent
assert (raiz / 'src' / 'run.py').exists(), 'Abra o notebook a partir da pasta do projeto.'
print('Projeto:', raiz)

# Descomente para executar a coleta real e o backtest:
# subprocess.run([sys.executable, '-m', 'src.run'], cwd=raiz, check=True)


## 3. Ler resultados do backtest

Estes dados só existem após a execução do motor. Não são gerados automaticamente com valores hipotéticos.


In [ ]:
import pandas as pd
import json
from IPython.display import display

arquivo = raiz / 'output' / 'backtest_results.json'
if not arquivo.exists():
    print('Ainda não há resultado real. Execute python -m src.run na pasta raiz.')
else:
    resultado = json.loads(arquivo.read_text(encoding='utf-8'))
    met = pd.Series(resultado['metrics'], name='Métrica')
    display(met.to_frame())
    print('Universo:', resultado['quality']['universe_mode'])


## 4. Rentabilidade acumulada: fundo, CDI e Ibovespa

A visualização abaixo lê exclusivamente o JSON do motor. A base de cada série é **100** no início do período de comparação. A rentabilidade é líquida de custos lineares de transação, porém **sem** taxas de administração, performance e tributos.


In [ ]:
if arquivo.exists():
    import matplotlib.pyplot as plt
    historico = pd.DataFrame(resultado['history'])
    historico['date'] = pd.to_datetime(historico['date'])
    historico = historico.set_index('date')
    ax = historico[['fund', 'cdi', 'ibov']].rename(columns={'fund':'Impact Found','cdi':'CDI','ibov':'Ibovespa'}).plot(figsize=(12,5))
    ax.set_title('ImpactUFSCar Found — Backtest histórico (base 100)')
    ax.set_ylabel('Índice base 100')
    ax.set_xlabel('Data')
    ax.grid(alpha=.2)
    plt.show()


## 5. Quais ações o modelo escolheu?

As posições são **resultado do ranking e das restrições**, não uma lista inventada. `score` representa a combinação de percentis, `notional_brl` representa o valor-alvo, e `adv_brl` representa a liquidez média financeira de 21 sessões.


In [ ]:
if arquivo.exists():
    carteira = pd.DataFrame(resultado['latest_holdings'])
    colunas = ['ticker','sector','momentum','volatility','score','weight','notional_brl','adv_brl']
    display(carteira[colunas].sort_values('score',ascending=False).reset_index(drop=True))
    print('Posições:',len(carteira),'| Peso total:',carteira.weight.sum())


## 6. Testes de robustez e desenvolvimento futuro

1. Reexecutar com `--momentum-weight 1.00` e `--momentum-weight 0.50` para verificar sensibilidade, **sem** escolher o melhor peso olhando para o período final de teste.
2. Reexecutar com custos de 20 e 40 bps para avaliar impacto do turnover.
3. Implementar universo **point-in-time** completo do IBrX 100, inclusive ações que saíram da bolsa.
4. Separar pré-2023 como desenvolvimento, 2023–2024 como validação e 2025–2026 como teste *out-of-sample* (divisão **ilustrativa**, a ser formalizada antes dos testes reais).
5. Acrescentar taxa de gestão/performance e estudo de capacidade do fundo de R$ 100 milhões.
6. Registrar todos os parâmetros e versões de dados para permitir replicação independente.

**Atenção:** um backtest positivo não garante sucesso futuro; é preciso confrontar o risco, custos, exposição setorial e vieses de seleção.
